# Transformer

The autoregressive transformer model is a flexible and deterministic
surrogate.

## When this might be a good fit

The transformer is a great option when a deterministic surrogate
suffices. Furthermore, it is more flexible than an MLP, supporting
conditioning on both static inputs and sequences. Additionally, because
it is autoregressive it naturally supports variable length outputs and
starting generation from a variable number of observed datapoints.

Unfortunately, Transformer surrogates are slower to train and infer with
and often require more tuning to match the performance of an MLP.
However, they tend to benefit more from scaling data or the size of the
network.

## How transformers work

### Architectural walkthrough

The transformer architecture is substantially more complex than an MLP’s
so we won’t cover it in great depth here.

The most important part of the Transformer architecture is the
*self-attention* mechansim which enables the model to process sequence
data in a highly expressive way. In an attention computation, the
entries of an incoming sequence $x_1, x_2, \dots, x_n$ are given
corresponding *query*, *key* and *value* vectors via projection matrices
(e.g. $q_1=W_{q}x_1^T$). These entries are collected in a $Q$ (query),
$K$ (key) and $V$ (value) matrix and the transformed sequence is
computed via $y=\mathrm{softmax}(\frac{QK^T}{\sqrt{d}})V$. In an
autoregressive transformer this computation is restricted so that query
vectors may only *attend* to key vectors from the past:

<figure id="fig-causal-attention">

<figcaption>Figure 1: Causal self-attention: each input may attend to
itself and earlier inputs, but not future inputs.</figcaption>
</figure>

> **Note**
>
> In practice *multi-head* attention is used, where models have several
> projection matrices and perform attention over the different sets of
> projections independently and in parallel and then combine the
> results.
>
> <figure id="fig-multi-head-attention">
> 
> <figcaption>Figure 2: Multi-head attention: Each block is a projected
> sequence of query, key, or value vectors.</figcaption>
> </figure>

During training the attention computation can be done over entire
sequences at once (this is a major strength of the transformer),
however, during inference, the sequence is iteratively generated. Since
the latest entry in the sequence still needs to attend to every entry
that came before it we are forced to either repeatedly recompute prior
*key* and *value* vectors or to store them in a *Key-Value Cache*
(KV-Cache). For larger transformers or generating longer sequences using
a KV-Cache can significantly speed up inference.

In Metis we also support options like interleaving full self-attention
with *sliding window* attention which halves the maximum size of the
attention window for a computational speedup:

<figure id="fig-sliding-window-attention">

<figcaption>Figure 3: Sliding window self-attention: Later tokens may
not attend to the furtherest tokens in the past.</figcaption>
</figure>

We additionally support specifying fewer key-value heads
(i.e. projections) than query heads to make attention more efficient and
reduce the size of the KV-cache. This is known as *Grouped-Query
Attention*.

<figure id="fig-grouped-query-attention">

<figcaption>Figure 4: Grouped-query attention: Multiple query heads
share each key-value head.</figcaption>
</figure>

### Training

Language transformers train with *Cross-Entropy Loss* over tokens but
since our data is typically continuous, we use a next-token MSE loss.

## Configuring a Transformer

The Transformer architecture is configured via these variables:

-   `sequence_len`
-   `x_input_dim`
-   `static_input_dim`
-   `n_layer`: The number of transformer blocks in the model
-   `n_head`: The number of query heads in each attention layer
-   `n_kv_head`: The number of key and value heads in each attention
    layer
-   `rope_base`: Base frequency for the rotary positional embeddings,
    don’t alter unless you really need to
-   `n_embd`: Hidden dimension of the network (controls width)
-   `in_and_out_biases`: Whether the input and output projection layers
    use learned biases
-   `window_pattern`: Sliding window attention pattern string. ‘L’
    (Long) for full context and ‘S’ (Short) for half context. Final
    layer is always ‘L’. (e.g. ‘SL’ would alternate between short and
    long context)
-   `cross_attn`: The cross attention config for sequence conditioning,
    can be `None`

The `sequence_len`, `x_input_dim` and `static_input_dim` are all
automatically configured based on the dataset config.

Within Metis we provide Transformer size presets:

| Config | `n_layer` | `n_head` | `n_kv_head` | `n_embd` | `window_pattern` |
|----------|------------:|------------:|------------:|------------:|----------|
| `transformer_small` | 6 | 4 | 4 | 64 | `L` |
| `transformer_default` | 12 | 8 | 8 | 128 | `L` |
| `transformer_large` | 16 | 16 | 16 | 256 | `L` |

Whilst the standard configs exclusively make use of full attention you
may find it beneficial use an interleaved pattern for computational
reasons. Similarly, it may be helpful to use more query heads than kv
heads (i.e. Grouped Query Attention).

``` yaml
defaults:
  - transformer_default
  - _self_

n_layer: ...
n_head: ... 
n_kv_head: ...
rope_base: ...
n_embd: ...
in_and_out_biases: ...
window_pattern: ...
cross_attn: ...
```